# RSSM Experiment on Colab
Runs the original DreamerV1 RSSM world model on `Pendulum-v1`.

## 1. Mount Google Drive (optional, to persist logs)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone the repo

In [ ]:
import os

REPO_URL = "https://github.com/YOUR_USERNAME/dreamer.git"  # TODO: replace with your repo URL
BRANCH   = "linux-colab-support"
WORKDIR  = "/content/dreamer"

if not os.path.exists(WORKDIR):
    !git clone -b {BRANCH} {REPO_URL} {WORKDIR}
os.chdir(WORKDIR)
print("Working directory:", os.getcwd())

## 3. Install dependencies

In [ ]:
!pip install -q tensorflow==2.8.0 tensorflow-probability==0.16.0
!pip install -q "gym==0.25.2" Pillow matplotlib "numpy<2"

## 4. Verify GPU

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print("GPUs:", gpus if gpus else "None — running on CPU")

## 5. Set log directory
Change `USE_DRIVE` to `True` to persist logs to Google Drive across sessions.

In [ ]:
USE_DRIVE = False  # set True to save logs to Drive

if USE_DRIVE:
    LOGDIR = "/content/drive/MyDrive/dreamer_logs/rssm"
else:
    LOGDIR = "/content/dreamer/logs/rssm"

!mkdir -p {LOGDIR}
print("Log directory:", LOGDIR)

## 6. Train RSSM

In [ ]:
!python dreamer.py \
    --task         gym_Pendulum-v1 \
    --world_model  rssm \
    --logdir       {LOGDIR} \
    --steps        50000 \
    --prefill      1000 \
    --time_limit   200 \
    --action_repeat 2 \
    --eval_every   5000 \
    --seed         42 \
    --precision    16 \
    --log_images   False

## 7. Plot results

In [ ]:
RESULTS_DIR = LOGDIR.replace("/rssm", "/results")
!mkdir -p {RESULTS_DIR}

!python evaluate.py \
    --logdirs      {LOGDIR} \
    --labels       RSSM \
    --world_models rssm \
    --outdir       {RESULTS_DIR} \
    --horizon      5 \
    --mse_transitions 1000

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

for fname in ["reward_curves.png", "prediction_mse.png"]:
    fpath = os.path.join(RESULTS_DIR, fname)
    if os.path.exists(fpath):
        plt.figure(figsize=(8, 4))
        plt.imshow(mpimg.imread(fpath))
        plt.axis('off')
        plt.title(fname)
        plt.show()